## EDA: 糖尿病予測チャレンジ
このノートブックでは、糖尿病予測チャレンジのデータを探索的に分析します。

In [16]:
import sys

sys.path.append("../utils")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge, Lasso
from scipy.stats import skew
from scipy.special import boxcox1p
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")

from utils import check_df

In [40]:
class CFG:
    seed = 42
    n_splits = 5
    target_col = "Heart Disease"

In [36]:
# データ読み込み
class Paths:
    p = "/Users/shirokoshikentaro/Desktop/Predicting Heart Disease/Data"
    train = p + "/train.csv"
    test = p + "/test.csv"
    sample = p + "/sample_submission.csv"

In [58]:
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)
sample_submission = pd.read_csv(Paths.sample)

# 元のデータのobject型の列を確認（reduce_mem_usage実行前）
print("元のデータのobject型の列:")
print(train.select_dtypes(include=["object"]).columns.tolist())

元のデータのobject型の列:
['Heart Disease']


In [20]:
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print("Memory usage of dataframe is {:.2f} MB".format(start_mem))

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if (
                    c_min > np.finfo(np.float16).min
                    and c_max < np.finfo(np.float16).max
                ):
                    df[col] = df[col].astype(np.float16)
                elif (
                    c_min > np.finfo(np.float32).min
                    and c_max < np.finfo(np.float32).max
                ):
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype("category")

    end_mem = df.memory_usage().sum() / 1024**2
    print("Memory usage after optimization is: {:.2f} MB".format(end_mem))
    print("Decreased by {:.1f}%".format(100 * (start_mem - end_mem) / start_mem))

    return df

In [21]:
train = reduce_mem_usage(train)

Memory usage of dataframe is 72.10 MB
Memory usage after optimization is: 13.22 MB
Decreased by 81.7%


In [22]:
check_df(train)

,Column,dtypes,NaN Count,Nunique,Unique Values
0,id,int32,0,630000,> 10 unique values
1,Age,int8,0,42,> 10 unique values
2,Sex,int8,0,2,"[1, 0]"
3,Chest pain type,int8,0,4,"[4, 1, 2, 3]"
4,BP,int16,0,66,> 10 unique values
5,Cholesterol,int16,0,150,> 10 unique values
6,FBS over 120,int8,0,2,"[0, 1]"
7,EKG results,int8,0,3,"[0, 2, 1]"
8,Max HR,int16,0,93,> 10 unique values
9,Exercise angina,int8,0,2,"[1, 0]"


In [23]:
print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")

train shape: (630000, 15)
test shape: (270000, 14)


In [24]:
# 数値型の列を取得
num_features = train.select_dtypes(include=[np.number]).columns.tolist()
num_features.remove("id")
# num_features.remove("Heart Disease")
num_features

['Age',
 'Sex',
 'Chest pain type',
 'BP',
 'Cholesterol',
 'FBS over 120',
 'EKG results',
 'Max HR',
 'Exercise angina',
 'ST depression',
 'Slope of ST',
 'Number of vessels fluro',
 'Thallium']

In [25]:
# カテゴリカル変数を抽出（object型とcategory型の両方を含む）
# reduce_mem_usageでobject型がcategory型に変換されているため、両方を含める
cat_features = train.select_dtypes(include=["object", "category"]).columns.tolist()
cat_features

['Heart Disease']

In [42]:
# =============================================================================
# Target Encoding
# =============================================================================
if train[CFG.target_col].dtype == "object":
    train[CFG.target_col] = train[CFG.target_col].map({"No": 0, "Yes": 1})

print(f"Target distribution:\n{train[CFG.target_col].value_counts(normalize=True)}")

Target distribution:
Heart Disease
Absence     0.55166
Presence    0.44834
Name: proportion, dtype: float64


In [45]:
# =============================================================================
# Feature Engineering（交互作用特徴量）
# =============================================================================
def add_features(df):
    """train / test に同一の特徴量変換を適用する関数"""

    # ------------------------------------------------------------------
    # 0) 型変換：int8 → int32 でオーバーフローを防止
    # ------------------------------------------------------------------
    num_cols = df.select_dtypes(include=["int8", "int16"]).columns
    for col in num_cols:
        df[col] = df[col].astype("int32")

    float_cols = df.select_dtypes(include=["float16"]).columns
    for col in float_cols:
        df[col] = df[col].astype("float32")

    # ------------------------------------------------------------------
    # 1) 数値 × 数値：医学的に意味のある組み合わせ
    # ------------------------------------------------------------------
    # 年齢 × 最大心拍数（高齢で心拍数が低い → 心機能低下の指標）
    df["Age_x_MaxHR"] = df["Age"] * df["Max HR"]
    df["Age_div_MaxHR"] = df["Age"] / (df["Max HR"] + 1e-5)

    # ST depression × 最大心拍数（運動負荷時の心臓反応を複合評価）
    df["STdep_x_MaxHR"] = df["ST depression"] * df["Max HR"]

    # 血圧 × コレステロール（心血管リスクの複合指標）
    df["BP_x_Chol"] = df["BP"] * df["Cholesterol"]

    # 年齢 × ST depression（高齢 + ST低下は重篤度が高い）
    df["Age_x_STdep"] = df["Age"] * df["ST depression"]

    # 年齢 × 血管数（高齢 + 血管異常の相乗効果）
    df["Age_x_Vessels"] = df["Age"] * df["Number of vessels fluro"]

    # 血圧 × 最大心拍数（心臓への負荷の総合指標）
    df["BP_x_MaxHR"] = df["BP"] * df["Max HR"]

    # コレステロール × 年齢
    df["Chol_x_Age"] = df["Cholesterol"] * df["Age"]

    # ST depression × 血管数（両方異常なら非常に高リスク）
    df["STdep_x_Vessels"] = df["ST depression"] * df["Number of vessels fluro"]

    # ------------------------------------------------------------------
    # 2) 数値 × カテゴリ：グループ別で数値の意味が変わる
    # ------------------------------------------------------------------
    # 性別ごとの最大心拍数（男女で正常値が異なる）
    df["Sex_x_MaxHR"] = df["Sex"] * df["Max HR"]

    # 性別ごとのコレステロール
    df["Sex_x_Chol"] = df["Sex"] * df["Cholesterol"]

    # 胸痛タイプ × ST depression
    df["ChestPain_x_STdep"] = df["Chest pain type"] * df["ST depression"]

    # 胸痛タイプ × 最大心拍数
    df["ChestPain_x_MaxHR"] = df["Chest pain type"] * df["Max HR"]

    # 運動狭心症 × ST depression（両方異常なら高リスク）
    df["ExAngina_x_STdep"] = df["Exercise angina"] * df["ST depression"]

    # 運動狭心症 × 最大心拍数
    df["ExAngina_x_MaxHR"] = df["Exercise angina"] * df["Max HR"]

    # タリウム × 血管数（検査結果の複合評価）
    df["Thallium_x_Vessels"] = df["Thallium"] * df["Number of vessels fluro"]

    # Slope of ST × ST depression
    df["SlopeSTx_STdep"] = df["Slope of ST"] * df["ST depression"]

    # ------------------------------------------------------------------
    # 3) 差分・比率：相対的な位置を表す
    # ------------------------------------------------------------------
    # 年齢別の理論上最大心拍数(220-Age)と実測値の差
    df["MaxHR_reserve"] = (220 - df["Age"]) - df["Max HR"]

    # 最大心拍数の達成率（実測 / 理論最大）
    df["MaxHR_ratio"] = df["Max HR"] / (220 - df["Age"] + 1e-5)

    # ------------------------------------------------------------------
    # 4) カテゴリ × カテゴリ：組み合わせ（文字列結合で新カテゴリ化）
    # ------------------------------------------------------------------
    # 胸痛タイプ × 運動狭心症
    df["ChestPain_ExAngina"] = df["Chest pain type"].astype(str) + "_" + df["Exercise angina"].astype(str)

    # 胸痛タイプ × タリウム
    df["ChestPain_Thallium"] = df["Chest pain type"].astype(str) + "_" + df["Thallium"].astype(str)

    # Slope of ST × 運動狭心症
    df["SlopeST_ExAngina"] = df["Slope of ST"].astype(str) + "_" + df["Exercise angina"].astype(str)

    # FBS × 性別
    df["FBS_Sex"] = df["FBS over 120"].astype(str) + "_" + df["Sex"].astype(str)

    return df

# train / test 両方に適用
train = add_features(train)
test = add_features(test)


In [46]:
# =============================================================================
# カラム定義
# =============================================================================
drop_cols = ["id", CFG.target_col]
feature_cols = [c for c in train.columns if c not in drop_cols]

# カテゴリカル変数の特定（文字列結合で作ったものも含む）
cat_cols = train[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
print(f"\nFeature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols):
    tag = " [cat]" if col in cat_cols else ""
    print(f"  {i+1:2d}. {col}{tag}")
print(f"\nCategorical columns ({len(cat_cols)}): {cat_cols}")

# LightGBM native handling用にcategory型に変換
for col in cat_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

x_train = train[feature_cols]
y_train = train[CFG.target_col]
x_test = test[feature_cols]



Feature columns (36):
   1. Age
   2. Sex
   3. Chest pain type
   4. BP
   5. Cholesterol
   6. FBS over 120
   7. EKG results
   8. Max HR
   9. Exercise angina
  10. ST depression
  11. Slope of ST
  12. Number of vessels fluro
  13. Thallium
  14. Age_x_MaxHR
  15. Age_div_MaxHR
  16. STdep_x_MaxHR
  17. BP_x_Chol
  18. Age_x_STdep
  19. Age_x_Vessels
  20. BP_x_MaxHR
  21. Chol_x_Age
  22. STdep_x_Vessels
  23. Sex_x_MaxHR
  24. Sex_x_Chol
  25. ChestPain_x_STdep
  26. ChestPain_x_MaxHR
  27. ExAngina_x_STdep
  28. ExAngina_x_MaxHR
  29. Thallium_x_Vessels
  30. SlopeSTx_STdep
  31. MaxHR_reserve
  32. MaxHR_ratio
  33. ChestPain_ExAngina [cat]
  34. ChestPain_Thallium [cat]
  35. SlopeST_ExAngina [cat]
  36. FBS_Sex [cat]

Categorical columns (4): ['ChestPain_ExAngina', 'ChestPain_Thallium', 'SlopeST_ExAngina', 'FBS_Sex']


In [47]:
# =============================================================================
# LightGBM Parameters
# =============================================================================
params = {
    "objective": "binary",
    "metric": "auc",
    "verbosity": -1,
    "n_estimators": 10000,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "random_state": CFG.seed,
    "n_jobs": -1,
}


In [48]:
# =============================================================================
# Cross Validation
# =============================================================================
cv = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)

oof_preds = np.zeros(len(x_train))
test_preds = np.zeros(len(x_test))
models = []
metrics = []
imp = pd.DataFrame()

for nfold, (train_idx, val_idx) in enumerate(cv.split(x_train, y_train)):
    print("\n" + "=" * 50)
    print(f"Fold {nfold + 1} / {CFG.n_splits}")
    print("=" * 50)

    x_tr, y_tr = x_train.iloc[train_idx], y_train.iloc[train_idx]
    x_va, y_va = x_train.iloc[val_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        x_tr,
        y_tr,
        eval_set=[(x_va, y_va)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(200),
        ],
    )

    # OOF予測の保存
    y_va_pred = model.predict_proba(x_va)[:, 1]
    oof_preds[val_idx] = y_va_pred

    # テスト予測の蓄積（fold平均）
    test_preds += model.predict_proba(x_test)[:, 1] / CFG.n_splits

    # モデルの保存
    models.append(model)

    # fold別AUC
    auc_va = roc_auc_score(y_va, y_va_pred)
    metrics.append({"fold": nfold + 1, "auc": auc_va, "best_iter": model.best_iteration_})
    print(f"  Fold {nfold + 1} AUC: {auc_va:.6f} (best_iteration: {model.best_iteration_})")

    # Feature Importance
    _imp = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": model.feature_importances_,
            "nfold": nfold + 1,
        }
    )
    imp = pd.concat([imp, _imp], axis=0, ignore_index=True)


Fold 1 / 5
[200]	valid_0's auc: 0.955085
[400]	valid_0's auc: 0.955294
[600]	valid_0's auc: 0.955332
  Fold 1 AUC: 0.955340 (best_iteration: 583)

Fold 2 / 5
[200]	valid_0's auc: 0.954157
[400]	valid_0's auc: 0.954349
  Fold 2 AUC: 0.954374 (best_iteration: 493)

Fold 3 / 5
[200]	valid_0's auc: 0.954897
[400]	valid_0's auc: 0.955104
[600]	valid_0's auc: 0.955144
  Fold 3 AUC: 0.955150 (best_iteration: 537)

Fold 4 / 5
[200]	valid_0's auc: 0.954363
[400]	valid_0's auc: 0.954625
  Fold 4 AUC: 0.954636 (best_iteration: 462)

Fold 5 / 5
[200]	valid_0's auc: 0.955162
[400]	valid_0's auc: 0.955444
[600]	valid_0's auc: 0.955488
  Fold 5 AUC: 0.955491 (best_iteration: 663)


In [50]:
# =============================================================================
# Overall Results
# =============================================================================
metrics_df = pd.DataFrame(metrics)
oof_auc = roc_auc_score(y_train, oof_preds)

print("\n" + "=" * 50)
print("RESULTS SUMMARY")
print("=" * 50)
print(metrics_df.to_string(index=False))
print(f"\nMean Fold AUC:   {metrics_df['auc'].mean():.6f} ± {metrics_df['auc'].std():.6f}")
print(f"Overall OOF AUC: {oof_auc:.6f}  ← LBと比較すべき値")
print("=" * 50)


RESULTS SUMMARY
 fold      auc  best_iter
    1 0.955340        583
    2 0.954374        493
    3 0.955150        537
    4 0.954636        462
    5 0.955491        663

Mean Fold AUC:   0.954998 ± 0.000475
Overall OOF AUC: 0.954996  ← LBと比較すべき値


In [51]:
# =============================================================================
# Feature Importance（全特徴量を重要度順に表示）
# =============================================================================
imp_summary = imp.groupby("feature")["importance"].mean().sort_values(ascending=False)
print("\nFeature Importance (mean across folds):")
print(imp_summary.to_string())

# importance が低い特徴量の確認（削除候補）
low_imp = imp_summary[imp_summary < 10].index.tolist()
if low_imp:
    print(f"\n⚠️ 低重要度の特徴量（削除検討）: {low_imp}")


Feature Importance (mean across folds):
feature
Max HR                     1869.8
Age_div_MaxHR              1039.2
Cholesterol                1023.2
BP_x_MaxHR                  816.2
Chol_x_Age                  788.6
BP_x_Chol                   786.4
Sex_x_Chol                  739.6
ChestPain_x_MaxHR           713.8
Age_x_Vessels               640.2
ChestPain_Thallium          633.0
BP                          626.2
Age                         591.2
Age_x_MaxHR                 569.4
Sex_x_MaxHR                 525.0
Age_x_STdep                 512.8
MaxHR_ratio                 427.0
SlopeSTx_STdep              390.0
STdep_x_MaxHR               377.0
ChestPain_x_STdep           370.2
MaxHR_reserve               338.8
SlopeST_ExAngina            338.4
STdep_x_Vessels             306.6
ExAngina_x_MaxHR            264.6
EKG results                 236.8
Thallium_x_Vessels          211.8
ST depression               206.0
ChestPain_ExAngina          205.6
ExAngina_x_STdep            204.8

In [52]:
# =============================================================================
# OOF予測の保存（Stacking・アンサンブル用）
# =============================================================================
oof_df = pd.DataFrame({"id": train["id"], "oof_pred": oof_preds, "target": y_train})
oof_df.to_csv("oof_lgbm.csv", index=False)
print("\nOOF predictions saved to oof_lgbm.csv")


OOF predictions saved to oof_lgbm.csv


In [60]:
# =============================================================================
# Submission
# =============================================================================
submission = sample_submission.copy()
submission[CFG.target_col] = test_preds
submission[CFG.target_col] = submission[CFG.target_col].astype(float)

print(f"\nSubmission shape: {submission.shape}")
print(f"Submission dtypes:\n{submission.dtypes}")
print(f"Prediction range: [{submission[CFG.target_col].min():.6f}, {submission[CFG.target_col].max():.6f}]")
print(submission.head())

submission.to_csv("submission_lgbm01.csv", index=False)
print("\nSubmission saved to submission01_lgbm.csv")


Submission shape: (270000, 2)
Submission dtypes:
id                 int64
Heart Disease    float64
dtype: object
Prediction range: [0.000452, 0.999858]
       id  Heart Disease
0  630000       0.958316
1  630001       0.009161
2  630002       0.985819
3  630003       0.004266
4  630004       0.175198

Submission saved to submission01_lgbm.csv
